In [17]:
import random
import pandas as pd
import numpy as np

# =============================================================================
# 1.1 CONFIGURAÇÃO DE PARÂMETROS E ESCOPO DE SEGURANÇA
# Os limites foram baseados em normas técnicas (ASHRAE, NASA, Sutton)
# para que o sistema possa validar se os dados recebidos são seguros.
# =============================================================================

LIMITES = {
    "temp_int": (21.0, 25.0),    
    "temp_ext": (-10.0, 38.0),   
    "energia_min": 80.0,
    "energia": (80.0, 100.0),         
    "pressao": (200.0, 300.0),    
    "integridade": 1,
    "modulos": 1
}

# Constantes de Engenharia Elétrica para cálculo de carga
CAPACIDADE_TOTAL = 50.0 # Capacidade nominal da bateria (kWh)
CONSUMO_ESTIMADO = 15.0 # Custo energético do primeiro impulso (kWh)
EFICIENCIA = 0.95 # Coeficiente de eficiência (considerando Perda Joule)

def simular_telemetria():
    #Simula a leitura de sensores (78% sucesso / 22% anomalia)
    sucesso = random.random() <= 0.78
    
    if sucesso:
        # Dados de sucesso
        dados = {
            "temp_int": round(random.uniform(*LIMITES["temp_int"]), 2),
            "temp_ext": round(random.uniform(*LIMITES["temp_ext"]), 2),
            "energia": round(random.uniform(*LIMITES["energia"]), 2),
            "pressao": round(random.uniform(*LIMITES["pressao"]), 2),
            "integridade": 1, 
            "modulos": 1     
        }
    else:
        # Dados de ANOMALIA (Fora das faixas seguras)
        dados = {
            "temp_int": round(random.uniform(26.0, 45.0), 2),
            "temp_ext": round(random.uniform(-40.0, -11.0), 2),
            "energia": round(random.uniform(0.0, 79.0), 2),
            "pressao": round(random.uniform(0.0, 199.0), 2),
            "integridade": random.choice([0, 1]),
            "modulos": 0      # Pelo menos um sistema crítico offline
        } 

    return dados

# =============================================================================
# 1.2 ALGORITMO DE VERIFICAÇÃO E LÓGICA GO/NO-GO
# Avaliação do vetor de telemetria contra os parâmetros de segurança.
# Aplica o critério de redundância zero: qualquer falha aborta a missão.
# =============================================================================

def verificar_decolagem(d):

    c1 = 21.0 <= d['temp_int'] <= 25.0
    c2 = -10.0 <= d['temp_ext'] <= 38.0
    c3 = d['energia'] >= 80.0
    c4 = 200.0 <= d['pressao'] <= 300.0
    c5 = d['integridade'] == 1
    c6 = d['modulos'] == 1
    
    return all([c1, c2, c3, c4, c5, c6])

# =============================================================================
# 1.3 GERAÇÃO DE DATASET PARA ANÁLISE PREDITIVA
# Processamento de dados em lote para validação estatística de autonomia.
# =============================================================================

def dataset_validacao(n=100):
    # Gera massa de dados histórica com cálculos de eficiência energética
    registros = []
    for i in range(n):
        d = simular_telemetria()

        # Cálculo da Autonomia Residual (Energia Atual * Eficiência - Consumo Inercial)
        energia_bruta = CAPACIDADE_TOTAL * (d['energia'] / 100)
        energia_liquida = energia_bruta * EFICIENCIA
        autonomia_residual = energia_liquida - CONSUMO_ESTIMADO

        # Inserção de métricas analíticas no registro
        d['id_missao'] = i + 1
        d['energia_bruta_kwh'] = round(energia_bruta, 2)
        d['autonomia_residual_kwh'] = round(autonomia_residual, 2)

        # Veredito final: Sistemas OK + Margem de Segurança > 10kWh
        d['go_no_go'] = "GO" if (verificar_decolagem(d) and autonomia_residual > 10) else "NO-GO"
        registros.append(d)

    return pd.DataFrame(registros)        

# =============================================================================
# 1.4 PROCESSAMENTO FINAL E RELATÓRIO DE MISSÃO
# Consolidação visual dos dados e veredito do sistema Aurora Siger.
# =============================================================================

# Dataset de 100 amostras
df_analises = dataset_validacao(100)

# 2. Extrai a 'Missão Atual' diretamente do dataset para garantir consistência
# iloc[0] pega a primeira linha da tabela gerada

missao_atual = df_analises.iloc[0]

# Define o veredito baseado na coluna 'go_no_go' do dataset
decolagem_ok = missao_atual['go_no_go'] == "GO"
status_texto = "[OK]" if decolagem_ok else "[FALHA]"

print("\n" + "="*55)
print(f"{'1. AMOSTRA DO DATASET DE ANÁLISE ENERGÉTICA':^55}")
print("="*55)
display(df_analises[['id_missao', 'energia', 'energia_bruta_kwh', 'autonomia_residual_kwh', 'go_no_go']].head(10))

print("\n" + "="*55)
print(f"{'2. DETALHAMENTO DA TELEMETRIA E AUTONOMIA':^55}")
print("="*55)

sensores = ["temp_int", "temp_ext", "energia", "pressao", "integridade", "modulos"]
for chave in sensores:
    valor = missao_atual[chave]    
    unidade = "°C" if "temp" in chave else "%" if "energia" in chave else "bar" if "pressao" in chave else ""
    print(f"{chave.replace('_', ' ').title():<15}: {valor:>7} {unidade:<5} -> {status_texto}")

print("-" * 55)

print(f"Energia Bruta  : {missao_atual['energia_bruta_kwh']:>7} kWh")
print(f"Autonomia Final: {missao_atual['autonomia_residual_kwh']:>7} kWh")

print("-" * 55)

# Estatísticas globais do lote de 100 missões
print(f"\nTaxa de Sucesso da Missão: {(df_analises['go_no_go'] == 'GO').sum()}%")
print(f"Média de Autonomia Residual: {df_analises['autonomia_residual_kwh'].mean():.2f} kWh")

print("-" * 55)

# Salva as missões em um arquivo CSV
df_analises.to_csv('telemetria_aurora.csv', index=False, sep=';', encoding='utf-8')

# Veredito final

if decolagem_ok:
    print(f"{'STATUS FINAL: [PRONTO PARA DECOLAR]':^55}")
else:
    print(f"{'STATUS FINAL: [DECOLAGEM ABORTADA - ANOMALIA]':^55}")

print("="*55)


      1. AMOSTRA DO DATASET DE ANÁLISE ENERGÉTICA      


,id_missao,energia,energia_bruta_kwh,autonomia_residual_kwh,go_no_go
0,1,92.09,46.05,28.74,GO
1,2,87.23,43.62,26.43,GO
2,3,83.80,41.90,24.80,GO
3,4,94.75,47.38,30.01,GO
4,5,94.84,47.42,30.05,GO
5,6,86.47,43.23,26.07,GO
6,7,85.49,42.74,25.61,GO
7,8,8.13,4.07,-11.14,NO-GO
8,9,85.54,42.77,25.63,GO
9,10,3.43,1.72,-13.37,NO-GO



       2. DETALHAMENTO DA TELEMETRIA E AUTONOMIA       
Temp Int       :   24.73 °C    -> [OK]
Temp Ext       :   10.11 °C    -> [OK]
Energia        :   92.09 %     -> [OK]
Pressao        :  273.49 bar   -> [OK]
Integridade    :       1       -> [OK]
Modulos        :       1       -> [OK]
-------------------------------------------------------
Energia Bruta  :   46.05 kWh
Autonomia Final:   28.74 kWh
-------------------------------------------------------

Taxa de Sucesso da Missão: 75%
Média de Autonomia Residual: 21.62 kWh
-------------------------------------------------------
          STATUS FINAL: [PRONTO PARA DECOLAR]          
